# Step 1: Setup & Data Preparation

In [1]:
# Cài đặt các thư viện Deep Learning cần thiết
# Lưu ý: Nếu chạy trên Terminal/CMD thì bỏ dấu '!' đi
#!pip install pandas numpy scikit-learn torch transformers datasets
!pip install accelerate -U

In [2]:
import torch
print("Có CUDA không?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Tên Card:", torch.cuda.get_device_name(0))
    print("Phiên bản CUDA:", torch.version.cuda)
else:
    print("Vẫn đang dùng CPU.")

Có CUDA không?: True
Tên Card: NVIDIA GeForce GTX 1650
Phiên bản CUDA: 11.8


In [ ]:
# 1. Load dữ liệu đã gán (SEED DATA) 
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Thiết bị đang sử dụng: {device}")


c:\Users\tran thien\.conda\envs\linear_regression\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Thiết bị đang sử dụng: cuda


In [4]:
# Load dữ liệu đã gán nhãn (424 dòng chuẩn)
df_labeled = pd.read_csv("training_data.csv")
#df_labeled.head()
#df_labeled.shape
file_path_unlabeled = 'unlabeled_data.csv'
df_unlabeled = pd.read_csv(file_path_unlabeled)
#df_unlabeled.shape    
df_unlabeled.head()

,id,text
0,Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0Xzg1Njk1MDM0MD...,"<person> thôi ra hp còn đỡ, mà mình hay ru..."
1,Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0XzE1MzQ0MTQ3NT...,tôi tận hưởng rồi xoài 1/² kg tôi tưởng một ký...
2,Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0XzE5MjQyNTMxNj...,<person> thuyệt tình à chứ
3,Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0Xzc3NTMxNDMxOD...,thà đi nước ngoài
4,Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0XzEwMTU5MDU3NT...,"<person> tin chuẩn chưa bạn yêu, nghèo có được..."


In [5]:
# Chuẩn bị class dataset cho pytorch
class HateSpeechDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        '''
        item = {
            'input_ids': Tensor([10, 20]),
            'attention_mask': Tensor([1, 1])
        }
        '''
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels:
            item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.encodings["input_ids"])
# tokenizer dữ liệu (biến chữ thành số)
model_name = "vinai/phobert-base-v2"   
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_labeled["text"].tolist(),
    df_labeled["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_labeled["label"].tolist()
)

train_encodings = tokenizer(train_texts, padding=True, truncation= True, max_length=128)
val_encodings = tokenizer(val_texts, padding=True, truncation = True, max_length = 128)

train_dataset = HateSpeechDataset(train_encodings, train_labels)
val_dataset = HateSpeechDataset(val_encodings, val_labels)


In [25]:
print(f"1. Số lượng tập Train: {len(val_texts)} dòng")
print(val_texts[:5])
print(f"\n2. Số lượng tập Validation: {len(val_labels)} dòng")
print(val_texts[:5])
# Xem thử mẫu đầu tiên trông như thế nào
print("\n--- Mẫu dữ liệu đầu tiên trong tập Train ---")
print(f"Câu văn (Text): {train_texts[0]}")
print(f"Nhãn (Label):   {train_labels[0]} (0=Không ghét, 1=Ghét)")



1. Số lượng tập Train: 85 dòng
['vô tới trong nam mà vẫn không tốt lên được à', 'sao không lên cao tốc mà chạy', 'lời nói của người chồng đau lòng người khác nhưng có ai nằm ở vị trí của người vợ không ạ', 'chị vợ ngoại tình cái lồn là sai nhưng phải có gì đó đánh đập chị vợ mới quá sợ không giám về', 'cho bọn ăn cơm này đi đảo hoang']

2. Số lượng tập Validation: 85 dòng
['vô tới trong nam mà vẫn không tốt lên được à', 'sao không lên cao tốc mà chạy', 'lời nói của người chồng đau lòng người khác nhưng có ai nằm ở vị trí của người vợ không ạ', 'chị vợ ngoại tình cái lồn là sai nhưng phải có gì đó đánh đập chị vợ mới quá sợ không giám về', 'cho bọn ăn cơm này đi đảo hoang']

--- Mẫu dữ liệu đầu tiên trong tập Train ---
Câu văn (Text): gửi clip này cho công an  giao thông điều tra để xử phạt vì phạm pháp luật !
Nhãn (Label):   0 (0=Không ghét, 1=Ghét)


In [7]:
# --- PHẦN 2: KIỂM TRA DỮ LIỆU SỐ HÓA (ENCODINGS) ---
print("\n" + "="*30)
print("CẤP ĐỘ 2: SAU KHI TOKENIZER (DANH SÁCH SỐ)")
print("="*30)
first_input_ids = train_encodings['input_ids'][0]
print(f"1. Độ dài của câu sau khi padding: {len(first_input_ids)} token")
print(f"2. Nội dung input_ids (10 số đầu): {first_input_ids[:10]} ...")
print("\n--- Dịch ngược (Decode) để kiểm tra ---")
decoded_text = tokenizer.decode(first_input_ids)
print(f"Kết quả dịch lại: {decoded_text}")


CẤP ĐỘ 2: SAU KHI TOKENIZER (DANH SÁCH SỐ)
1. Độ dài của câu sau khi padding: 128 token
2. Nội dung input_ids (10 số đầu): [0, 409, 1429, 23, 13, 675, 1510, 574, 2178, 184] ...

--- Dịch ngược (Decode) để kiểm tra ---
Kết quả dịch lại: <s> gửi clip này cho công an giao thông điều tra để xử phạt vì phạm pháp luật ! </s> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>


# Huấn luyện Model (Fine-tunning PhoBERT)

In [8]:
# 1. Load model PhoBERT 
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3).to(device)

# 2. Cấu hình tham số huấn luyện
training_args = TrainingArguments(
    output_dir = './results',
    num_train_epochs = 5,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 16,
    warmup_steps = 50,
    weight_decay = 0.01,
    logging_dir = './logs',
    logging_steps = 10,
    eval_strategy = "epoch",
    save_strategy = "no"
)
# 3. Khởi tạo Trainer 
trainer = Trainer(
    model = model, 
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)
trainer.train()
print("Huấn luyện hoàn tất!")


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.874500,0.886197
2,0.846000,0.798805
3,0.527600,0.823606
4,0.466500,0.785201
5,0.308100,0.774441


Huấn luyện hoàn tất!


In [9]:
print(" Đang dự đoán cho dữ liệu chưa gán nhãn...")

# Tokenizer cho dữ liệu chưa gán nhãn
unlabeled_texts = df_unlabeled['text'].tolist()
unlabeled_encodings = tokenizer(unlabeled_texts, padding=True, truncation=True, max_length=128)
unlabeled_dataset = HateSpeechDataset(unlabeled_encodings)

# 2. Chạy dự đoán
predictions = trainer.predict(unlabeled_dataset)

# 3. Xử lý kết quả dự đoán
preds = predictions.predictions
probs = torch.nn.functional.softmax(torch.tensor(preds), dim=-1)

# Lấy nhãn có xác xuất cao nhấtt
final_labels = np.argmax(preds, axis = 1)
# Lấy độ tự tin (confidence Score)
confidence_scores = np.max(probs.numpy(), axis=1)
# Gán ngược lại vào DataFrame
df_unlabeled['predicted_label'] = final_labels
df_unlabeled['confidence'] = confidence_scores

print("✅ Đã dự đoán xong!")
print(df_unlabeled.head(10)) # Xem thử 10 dòng đầu


 Đang dự đoán cho dữ liệu chưa gán nhãn...


✅ Đã dự đoán xong!
                                                  id  \
0  Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0Xzg1Njk1MDM0MD...   
1  Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0XzE1MzQ0MTQ3NT...   
2  Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0XzE5MjQyNTMxNj...   
3  Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0Xzc3NTMxNDMxOD...   
4  Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0XzEwMTU5MDU3NT...   
5  Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0XzE1NTAzMjIwOT...   
6  Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0XzE5MDEyOTIwMj...   
7  Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0XzE1MzUxODU3NT...   
8  Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0XzE2MzcxMDMyNz...   
9  Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0XzE4NzE3OTQwNz...   

                                                text  predicted_label  \
0  <person> thôi ra hp còn đỡ, mà mình hay ru...                0   
1  tôi tận hưởng rồi xoài 1/² kg tôi tưởng một ký...                0   
2                         <person> thuyệt tình à chứ                0   
3                                  thà đi nước ngoài                0   

# 5. Xuất file kết quả & lọc câu hỏi

In [10]:
# 1. Lưu toàn bộ kết quả
df_unlabeled.to_csv('du_lieu_da_gan_nhan_tu_dong.csv', index=False)
print("💾 Đã lưu file: du_lieu_da_gan_nhan_tu_dong.csv")

# 2. Lọc ra những câu Model "lưỡng lự" nhất (Độ tự tin < 0.6)
# Đây là những câu bạn CẦN KIỂM TRA LẠI bằng tay
low_confidence_df = df_unlabeled[df_unlabeled['confidence'] < 0.6].sort_values(by='confidence')
low_confidence_df.to_csv('can_check_tay.csv', index=False)

print(f"⚠️ Có {len(low_confidence_df)} câu có độ tin cậy thấp cần bạn check tay trong file 'can_check_tay.csv'")

💾 Đã lưu file: du_lieu_da_gan_nhan_tu_dong.csv
⚠️ Có 1820 câu có độ tin cậy thấp cần bạn check tay trong file 'can_check_tay.csv'
